# 📄 RAG sobre PDF con HuggingFace + FAISS
### Modelo de embeddings: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`

Este notebook implementa un sistema RAG (Retrieval-Augmented Generation) completo:
1. Carga y parsea un PDF
2. Divide el texto en chunks
3. Genera embeddings con el modelo multilingüe de HuggingFace
4. Almacena los vectores en FAISS
5. Responde preguntas usando recuperación semántica + LLM

## 1. 📦 Instalación de dependencias

In [ ]:
!pip install -q langchain langchain-community langchain-huggingface
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q pypdf
!pip install -q groq  # LLM gratuito (usaremos Groq)
!pip install -q langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 2.2 MB/s eta 0:00:00


## 2. 📂 Cargar el PDF

Podés subir tu PDF directamente desde tu computadora.

In [ ]:
from google.colab import files

print("Seleccioná tu archivo PDF:")
uploaded = files.upload()

# Obtener el nombre del archivo subido
pdf_filename = list(uploaded.keys())[0]
print(f"\n✅ Archivo cargado: {pdf_filename}")

Seleccioná tu archivo PDF:


Saving archlinux-install.md to archlinux-install.md

✅ Archivo cargado: archlinux-install.md


## 3. 📃 Extraer y dividir el texto del PDF

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Cargar el PDF
loader = PyPDFLoader(pdf_filename)
pages = loader.load()

print(f"📄 Páginas cargadas: {len(pages)}")
print(f"\n--- Vista previa de la primera página ---")
print(pages[0].page_content[:500])

ModuleNotFoundError: No module named 'langchain.text_splitter'

In [ ]:
# Dividir el texto en chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # Tamaño de cada chunk en caracteres
    chunk_overlap=50,     # Superposición entre chunks para no perder contexto
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = text_splitter.split_documents(pages)

print(f"✅ Total de chunks generados: {len(chunks)}")
print(f"\n--- Ejemplo de chunk ---")
print(chunks[0].page_content)

## 4. 🧠 Generar Embeddings con HuggingFace

Usamos el modelo `paraphrase-multilingual-MiniLM-L12-v2`, que soporta español y más de 50 idiomas.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

print("⏳ Cargando modelo de embeddings... (puede tardar unos segundos)")

embeddings_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": "cpu"},  # Cambiá a 'cuda' si tenés GPU disponible
    encode_kwargs={"normalize_embeddings": True}  # Normalizar mejora la similitud coseno
)

print("✅ Modelo de embeddings cargado correctamente")

# Verificar dimensiones del modelo
test_embedding = embeddings_model.embed_query("texto de prueba")
print(f"📐 Dimensiones del embedding: {len(test_embedding)}")

## 5. 🗄️ Crear la base de vectores con FAISS

In [ ]:
from langchain_community.vectorstores import FAISS

print("⏳ Generando embeddings e indexando en FAISS...")

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings_model
)

print(f"✅ Vector store creado con {len(chunks)} chunks")

# Guardar el índice localmente (opcional)
vectorstore.save_local("faiss_index")
print("💾 Índice FAISS guardado en './faiss_index'")

## 6. 🔍 Probar la búsqueda semántica (retrieval)

In [ ]:
# Probar que el retrieval funciona correctamente
query_prueba = "¿De qué trata el documento?"  # Modificá según tu PDF

docs_relevantes = vectorstore.similarity_search(query_prueba, k=3)

print(f"🔎 Pregunta: {query_prueba}")
print(f"\n📚 Top {len(docs_relevantes)} chunks más relevantes:\n")
for i, doc in enumerate(docs_relevantes):
    print(f"--- Chunk {i+1} (página {doc.metadata.get('page', '?') + 1}) ---")
    print(doc.page_content)
    print()

## 7. 🤖 Configurar el LLM (Groq - gratuito)

Usamos **Groq** como LLM por su API gratuita y alta velocidad.

👉 Obtené tu API key gratis en: https://console.groq.com

También podés usar OpenAI, Anthropic, u otro LLM de tu preferencia.

In [ ]:
import os
from google.colab import userdata

# Opción 1: Usar Colab Secrets (recomendado)
# Guardá tu key en: Colab → ícono 🔑 → Agregar secreto con nombre GROQ_API_KEY
try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    print("✅ API key cargada desde Colab Secrets")
except:
    # Opción 2: Ingresar manualmente
    import getpass
    os.environ["GROQ_API_KEY"] = getpass.getpass("Ingresá tu Groq API key: ")
    print("✅ API key configurada")

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",  # Modelo rápido y gratuito
    temperature=0.1,               # Baja temperatura para respuestas más precisas
    max_tokens=1024
)

print("✅ LLM (Groq / LLaMA 3.1) configurado")

## 8. 🔗 Armar el pipeline RAG completo

In [ ]:
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# Prompt personalizado en español
prompt_template = """
Eres un asistente experto que responde preguntas basándose ÚNICAMENTE en el contexto proporcionado.
Si la respuesta no se encuentra en el contexto, decí "No encontré información sobre eso en el documento".
Responde siempre en el mismo idioma que la pregunta.

Contexto:
{context}

Pregunta: {question}

Respuesta:"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

# Crear el retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}  # Recuperar los 4 chunks más relevantes
)

# Armar la cadena RAG
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",  # Concatena todos los chunks en el prompt
    retriever=retriever,
    return_source_documents=True,  # Devuelve los chunks usados
    chain_type_kwargs={"prompt": PROMPT}
)

print("✅ Pipeline RAG listo")

## 9. 💬 ¡Hacé preguntas sobre tu PDF!

In [ ]:
def preguntar(pregunta: str, mostrar_fuentes: bool = True):
    """Función para hacer preguntas al RAG y mostrar la respuesta."""
    print(f"\n❓ Pregunta: {pregunta}")
    print("-" * 60)

    resultado = rag_chain.invoke({"query": pregunta})

    print(f"💬 Respuesta:\n{resultado['result']}")

    if mostrar_fuentes:
        print(f"\n📚 Fuentes utilizadas ({len(resultado['source_documents'])} chunks):")
        for i, doc in enumerate(resultado['source_documents']):
            pagina = doc.metadata.get('page', '?')
            print(f"  [{i+1}] Página {pagina + 1}: {doc.page_content[:150]}...")

    return resultado

In [ ]:
# ✏️ Modificá la pregunta según el contenido de tu PDF
resultado = preguntar("¿De qué trata el documento?")

In [ ]:
# Hacé más preguntas
resultado = preguntar("¿Cuáles son los puntos más importantes?")

## 10. 💡 Modo interactivo (loop de preguntas)

In [ ]:
print("🤖 Chat RAG interactivo — escribí 'salir' para terminar\n")

while True:
    pregunta = input("\nTu pregunta: ").strip()

    if pregunta.lower() in ["salir", "exit", "quit", "q"]:
        print("👋 ¡Hasta luego!")
        break

    if not pregunta:
        print("Por favor, ingresá una pregunta.")
        continue

    preguntar(pregunta, mostrar_fuentes=False)

---
## 📝 Notas y personalización

| Parámetro | Dónde | Qué hace |
|-----------|-------|----------|
| `chunk_size` | Celda 3 | Tamaño de cada fragmento de texto |
| `chunk_overlap` | Celda 3 | Superposición entre chunks |
| `k` en retriever | Celda 8 | Cuántos chunks recuperar por pregunta |
| `temperature` | Celda 6 | 0 = más preciso, 1 = más creativo |
| `model` en ChatGroq | Celda 6 | Cambiá a `llama-3.3-70b-versatile` para mayor calidad |

### LLMs alternativos gratuitos
- **Groq**: `llama-3.1-8b-instant`, `mixtral-8x7b-32768`, `llama-3.3-70b-versatile`
- **Ollama local**: Ejecutar LLMs en tu máquina sin API key
- **Google Gemini**: Via `langchain-google-genai`